# HallusionBench — Google Colab End-to-End Runner (Nhóm 2: Đủ Phần A + B + C trên 3 Mô hình)

**Mục đích:** Kiểm chứng **THẬT** toàn bộ pipeline **A → B → C** trên dataset **HallusionBench thật** và **3 mô hình thật** (`SmolVLM2-2.2B-Instruct`, `Qwen3-VL-4B-Instruct`, `Phi-4-multimodal-instruct`):
- **Cell 1:** Mount Google Drive (lưu kết quả bền vững) + kiểm tra GPU / VRAM / BF16 native.
- **Cell 2:** Clone nhánh `benchmark/hallusionbench` + cài thư viện.
- **Cell 3:** Chạy toàn bộ **35 unit tests** (A + B + C + E2E) & kiểm tra cấu trúc repo.
- **Cell 4 (Phần A thật + Tự lưu Dataset lên Drive):** Lần đầu tự tải `rayguan/HallusionBench` từ Hugging Face và **lưu 1 bản `.tar` vĩnh viễn lên Google Drive**; các lần sau chỉ copy từ Drive ra SSD Colab trong ~2 giây (không phải tải lại). Sau đó chạy `audit` + xuất `pilot_ids.json`.
- **Cell 5 (Phần B thật):** Chạy inference thật cho từng model (`smoke_test` 2 mẫu → `pilot` dùng đúng `pilot_ids.json` của A → hoặc `full` khi sẵn sàng), tự lưu từng mẫu vào Drive & tự resume khi Colab ngắt.
- **Cell 6 (Phần C thật — Chấm điểm):** Chạy `scoring.py` chấm `aAcc / fAcc / qAcc / VD / VS / Easy / Hard`, kiểm chứng bằng `scripts/check_results.py` & `scripts/check_manifest.py`, in bảng so sánh 3 model.
- **Cell 7 (Phần C thật — Phân tích lỗi):** Chạy `analysis.py` trích xuất bảng ca lỗi 6 nhãn (`error_cases.md`) và tìm các ca 3 model trả lời khác nhau (`disagreements.json`).
- **Cell 8 (Lấy kết quả về máy):** Tự nén toàn bộ báo cáo audit, `pilot_ids.json`, 7 artifact của 3 model và bảng ca lỗi thành 1 file `.zip` (lưu trên Drive + tự bật tải xuống máy tính).

**Quy tắc chính sách bắt buộc:**
- **KHÔNG** gọi lượt chạy `float16` trên T4 hay lượt chạy chưa khóa revision là `official`.
- **KHÔNG** bật lượng tử hóa (`INT4`/`INT8`) và **KHÔNG** tự sửa `configs/`.

## Cell 1 — Kết nối Google Drive & kiểm tra GPU / CUDA / BF16 native

In [ ]:
import os
import sys
from pathlib import Path
import torch
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/hallusionbench_runs")
OUTPUT_DIR = DRIVE_ROOT / "results"
AUDIT_DIR = DRIVE_ROOT / "audit_and_pilot"
DRIVE_DATASET_CACHE = DRIVE_ROOT / "dataset_cache"
LOCAL_DATA_ROOT = Path("/content/data/HallusionBench")
LOCAL_HF_CACHE = Path("/content/hf_cache")

for p in (OUTPUT_DIR, AUDIT_DIR, DRIVE_DATASET_CACHE, LOCAL_DATA_ROOT.parent, LOCAL_HF_CACHE):
    p.mkdir(parents=True, exist_ok=True)

os.environ["DATA_ROOT"] = str(LOCAL_DATA_ROOT)
os.environ["HF_HOME"] = str(LOCAL_HF_CACHE)
os.environ["MODEL_CACHE_DIR"] = str(LOCAL_HF_CACHE)
os.environ["OUTPUT_DIR"] = str(OUTPUT_DIR)
os.environ["AUDIT_DIR"] = str(AUDIT_DIR)
os.environ["DRIVE_DATASET_CACHE"] = str(DRIVE_DATASET_CACHE)

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

bf16_emulated = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False
try:
    bf16_native = torch.cuda.is_bf16_supported(including_emulation=False) if torch.cuda.is_available() else False
except TypeError:
    bf16_native = bf16_emulated and (torch.cuda.get_device_properties(0).major >= 8)

print(f"BF16 native supported (hardware): {bf16_native} | BF16 emulated: {bf16_emulated}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | sm_{props.major}{props.minor} | {round(props.total_memory / 2**30, 2)} GiB")

print("\nOUTPUT_DIR (lưu bền vững trên Drive):", OUTPUT_DIR)
if not torch.cuda.is_available():
    print("CẢNH BÁO: Chưa bật GPU! Vào Runtime -> Change runtime type -> chọn GPU (T4/L4/A100).")
elif not bf16_native:
    print("LƯU Ý: GPU này KHÔNG có BF16 native (vd: Tesla T4 sm_75). Chỉ dùng float16 cho smoke_test/pilot và gửi log này cho nhóm trưởng duyệt.")

!nvidia-smi

## Cell 2 — Clone nhánh `benchmark/hallusionbench` & cài đặt thư viện

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/hoainhan256/vlm-hallucination-benchmark.git"
BRANCH = "benchmark/hallusionbench"
REPO_DIR = Path("/content/vlm-hallucination-benchmark")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

%cd /content/vlm-hallucination-benchmark
!git rev-parse HEAD
!ls -la
!ls -la benchmarks/hallusionbench

missing_core = [p for p in ("vlm_benchmark", "benchmarks/hallusionbench/dataset.py", "benchmarks/hallusionbench/inference.py", "benchmarks/hallusionbench/scoring.py") if not Path(p).exists()]
if missing_core:
    raise RuntimeError(
        f"Nhánh {BRANCH} trên GitHub CHƯA có các file mới nhất: {missing_core}. "
        "Hãy chạy git add benchmarks/hallusionbench/ && git commit && git push origin benchmark/hallusionbench trên máy bạn trước, rồi chạy lại Cell 2!"
    )

!pip install -q -r requirements.txt
!pip install -q --upgrade transformers accelerate qwen-vl-utils num2words peft backoff soundfile

## Cell 3 — Kiểm tra cấu trúc repo & chạy toàn bộ 35 unit tests (A + B + C + E2E)

In [ ]:
import sys
from pathlib import Path

repo_root = str(Path("/content/vlm-hallucination-benchmark").resolve())
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from vlm_benchmark.config import validate_all

config_warnings = validate_all(
    "configs/models.yaml",
    "configs/datasets.yaml",
    "configs/experiment.yaml",
)
print("Trạng thái khóa configs chung (official gate):")
for item in config_warnings:
    print(" -", item)

!python scripts/check_repository.py
!python -m pytest -v tests/ benchmarks/hallusionbench/tests/

## Cell 4 (PHẦN A THẬT + LƯU VĨNH VIỄN LÊN GOOGLE DRIVE) — Tải/nạp dataset HallusionBench, chạy Audit & xuất `pilot_ids.json`

- **Lần chạy đầu tiên:** Tự tải dataset từ Hugging Face bằng mạng tốc độ cao của Colab rồi đóng gói thành **1 file `.tar` lưu vĩnh viễn trên Google Drive** (`/content/drive/MyDrive/hallusionbench_runs/dataset_cache/HallusionBench_2067dfdc.tar`).
- **Các lần chạy sau:** Tự giải nén từ file `.tar` trên Drive ra SSD local `/content/data/HallusionBench` trong ~2 giây — **không cần tải lại từ Hugging Face** và **không đọc ảnh chậm qua Drive FUSE**.

In [ ]:
import json
import os
import shutil
import tarfile
from pathlib import Path
from huggingface_hub import snapshot_download

HF_DATASET_REPO = "rayguan/HallusionBench"
HF_DATASET_REVISION = "2067dfdce5abc7efdbb0927f4721ae282f0b6c51"
LOCAL_DATA_ROOT = Path(os.environ["DATA_ROOT"])
AUDIT_DIR = Path(os.environ["AUDIT_DIR"])
DRIVE_DATASET_TAR = Path(os.environ["DRIVE_DATASET_CACHE"]) / f"HallusionBench_{HF_DATASET_REVISION[:8]}.tar"
AUDIT_REPORT = AUDIT_DIR / "hallusionbench_audit_report.json"
PILOT_FILE = AUDIT_DIR / "pilot_ids.json"

if not (LOCAL_DATA_ROOT / "HallusionBench.json").is_file():
    if DRIVE_DATASET_TAR.is_file():
        print(f"Đã tìm thấy dataset lưu sẵn trên Google Drive: {DRIVE_DATASET_TAR}")
        print(f"Đang giải nén nhanh ra SSD local {LOCAL_DATA_ROOT}...")
        if LOCAL_DATA_ROOT.exists():
            shutil.rmtree(LOCAL_DATA_ROOT)
        LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
        with tarfile.open(DRIVE_DATASET_TAR, "r") as tf:
            tf.extractall(LOCAL_DATA_ROOT)
        print("Giải nén xong từ Google Drive!")
    else:
        print(f"Lần đầu: Đang tải {HF_DATASET_REPO} @ {HF_DATASET_REVISION} về {LOCAL_DATA_ROOT}...")
        snapshot_download(
            repo_id=HF_DATASET_REPO,
            repo_type="dataset",
            revision=HF_DATASET_REVISION,
            local_dir=str(LOCAL_DATA_ROOT),
        )
        print(f"Đang lưu bản sao vĩnh viễn lên Google Drive: {DRIVE_DATASET_TAR}...")
        tmp_tar = Path("/content/HallusionBench_tmp.tar")
        with tarfile.open(tmp_tar, "w") as tf:
            for item in LOCAL_DATA_ROOT.iterdir():
                if item.name == ".cache":
                    continue
                tf.add(item, arcname=item.name)
        shutil.move(str(tmp_tar), str(DRIVE_DATASET_TAR))
        print("Đã lưu xong trên Google Drive! Từ các lần sau chỉ cần bấm chạy là tự lấy từ Drive.")
else:
    print("Dataset đã sẵn sàng ở SSD local:", LOCAL_DATA_ROOT)

os.environ["DATASET_REV"] = HF_DATASET_REVISION
os.environ["PILOT_FILE"] = str(PILOT_FILE)

# 1. Chạy Audit thật bằng Phần A (kiểm đếm dòng, ảnh tồn tại, ảnh hỏng PIL, lệch .PNG, SHA-256)
!python -m benchmarks.hallusionbench.dataset \
  --mode audit \
  --dataset-json "$DATA_ROOT/HallusionBench.json" \
  --data-root "$DATA_ROOT" \
  --dataset-source "rayguan/HallusionBench" \
  --dataset-revision "$DATASET_REV" \
  --output "$AUDIT_DIR/hallusionbench_audit_report.json"

# 2. Xuất & kiểm chứng danh sách Pilot ID cố định (giữ trọn nhóm/cặp để Phần C chấm fAcc & qAcc)
if not PILOT_FILE.is_file():
    !python -m benchmarks.hallusionbench.dataset \
      --mode export-pilot \
      --dataset-json "$DATA_ROOT/HallusionBench.json" \
      --data-root "$DATA_ROOT" \
      --dataset-source "rayguan/HallusionBench" \
      --dataset-revision "$DATASET_REV" \
      --target-groups 6 \
      --output "$AUDIT_DIR/pilot_ids.json"
else:
    print("Dùng lại pilot_ids.json đã khóa sẵn trên Drive:", PILOT_FILE)

!python -m benchmarks.hallusionbench.dataset \
  --mode verify-pilot \
  --dataset-json "$DATA_ROOT/HallusionBench.json" \
  --data-root "$DATA_ROOT" \
  --pilot-file "$AUDIT_DIR/pilot_ids.json"

report = json.loads(AUDIT_REPORT.read_text(encoding="utf-8"))
print("\n=== TÓM TẮT AUDIT THẬT (ĐIỀN VÀO DATASET_NOTES.md) ===")
print("json_sha256:", report["json_sha256"])
print("is_clean:", report["integrity"]["is_clean"])
print("counts:", json.dumps(report["counts"], indent=2))
print("distributions:", json.dumps(report["distributions"], indent=2))

## Cell 5 (PHẦN B THẬT) — Chạy Inference thật cho cả 3 mô hình (`SmolVLM2` → `Qwen3-VL` → `Phi-4`)

- Khuyến nghị chạy `STAGE = "smoke_test"` trước (2 mẫu/model ~ vài phút) để kiểm chứng nạp mô hình + GPU.
- Sau khi ổn, đổi `STAGE = "pilot"` để chạy **đúng cùng 1 tập `pilot_ids.json`** cho cả 3 mô hình.
- Muốn chạy 1 model riêng, chỉ cần sửa `MODELS_TO_RUN = ["smolvlm2"]`.

In [ ]:
import os
import subprocess
import torch

# === CHỌN PHẠM VI & MÔ HÌNH CHẠY ===
MODELS_TO_RUN = ["smolvlm2", "qwen3_vl", "phi4_multimodal"]
STAGE = "smoke_test"     # Đổi thành "pilot" để chạy tập pilot_ids.json trọn nhóm của A
MAX_NEW_TOKENS = 128
MAX_SAMPLES = 2          # Chỉ dùng khi STAGE == "smoke_test"

try:
    bf16_native = torch.cuda.is_available() and torch.cuda.is_bf16_supported(including_emulation=False)
except TypeError:
    bf16_native = torch.cuda.is_available() and torch.cuda.is_bf16_supported() and (torch.cuda.get_device_properties(0).major >= 8)

DTYPE = "bfloat16" if bf16_native else "float16"
gpu_tag = "bf16gpu" if bf16_native else "t4"
EXPERIMENT_ID = f"hb_colab_{gpu_tag}_{STAGE}"
os.environ["EXPERIMENT_ID"] = EXPERIMENT_ID
os.environ["DTYPE"] = DTYPE
os.environ["STAGE"] = STAGE

RUN_DIRS = {}
for model_key in MODELS_TO_RUN:
    attn_impl = "eager" if (model_key == "phi4_multimodal" and not bf16_native) else "sdpa"
    run_id = f"{STAGE}_{model_key}_{DTYPE}"
    run_dir = f"{os.environ['OUTPUT_DIR']}/{EXPERIMENT_ID}/{run_id}"
    RUN_DIRS[model_key] = run_dir

    cmd = [
        "python", "-m", "benchmarks.hallusionbench.inference",
        "--mode", "run",
        "--allow-model-load",
        "--model-key", model_key,
        "--stage", STAGE,
        "--dtype", DTYPE,
        "--attn-implementation", attn_impl,
        "--max-new-tokens", str(MAX_NEW_TOKENS),
        "--dataset-json", f"{os.environ['DATA_ROOT']}/HallusionBench.json",
        "--data-root", os.environ["DATA_ROOT"],
        "--dataset-revision", os.environ["DATASET_REV"],
        "--experiment-id", EXPERIMENT_ID,
        "--run-id", run_id,
        "--output-root", os.environ["OUTPUT_DIR"],
    ]
    if model_key == "phi4_multimodal":
        cmd.append("--trust-remote-code")
    if STAGE == "smoke_test":
        cmd.extend(["--max-samples", str(MAX_SAMPLES)])
    elif STAGE == "pilot":
        cmd.extend(["--pilot-file", os.environ["PILOT_FILE"]])

    print("\n" + "=" * 70)
    print(f"CHẠY MODEL: {model_key} | STAGE: {STAGE} | DTYPE: {DTYPE} | ATTN: {attn_impl}")
    print("Lệnh:", " ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    print(f"--> Kết thúc {model_key} với exit code = {rc} | Output: {run_dir}")

print("\nTất cả thư mục run:", RUN_DIRS)

## Cell 6 (PHẦN C THẬT) — Chấm điểm `aAcc / fAcc / qAcc`, kiểm chứng 7 artifact & in bảng 3 mô hình

In [ ]:
import json
import os
import subprocess
from pathlib import Path

summary_rows = []
for model_key, run_dir_str in RUN_DIRS.items():
    run_dir = Path(run_dir_str)
    print("\n" + "=" * 70)
    print(f"CHẤM ĐIỂM & KIỂM CHỨNG: {model_key} ({run_dir})")

    status_data = json.loads((run_dir / "run_status.json").read_text(encoding="utf-8"))
    meta_data = json.loads((run_dir / "run_metadata.json").read_text(encoding="utf-8"))

    if status_data.get("status") != "BLOCKED":
        subprocess.run(
            [
                "python", "-m", "benchmarks.hallusionbench.scoring",
                "--run-dir", str(run_dir),
                "--dataset-json", f"{os.environ['DATA_ROOT']}/HallusionBench.json",
                "--data-root", os.environ["DATA_ROOT"],
            ],
            check=True,
        )

    subprocess.run(["python", "scripts/check_results.py", str(run_dir / "predictions.jsonl")], check=True)
    subprocess.run(["python", "scripts/check_manifest.py", str(run_dir)], check=True)

    metrics_data = json.loads((run_dir / "metrics.json").read_text(encoding="utf-8"))
    m = metrics_data.get("metrics") or {}
    c = metrics_data.get("summary_counts") or {}
    summary_rows.append({
        "model": model_key,
        "status": status_data.get("status"),
        "dtype": meta_data.get("dtype"),
        "model_rev": meta_data.get("resolved_model_revision"),
        "scored_q": c.get("scored_questions", 0),
        "aAcc": m.get("aAcc"),
        "fAcc": m.get("fAcc"),
        "qAcc": m.get("qAcc"),
        "errors": c.get("error_records", 0),
    })

print("\n=== BẢNG TỔNG HỢP 3 MÔ HÌNH (THẬT — GHI RÕ STAGE:", os.environ["STAGE"], ") ===")
print("| Model | Status | Dtype | Scored Q | aAcc | fAcc | qAcc | Errors | Model Rev |")
print("|---|---|---|---:|---:|---:|---:|---:|---|")
for r in summary_rows:
    print(
        f"| `{r['model']}` | `{r['status']}` | `{r['dtype']}` | {r['scored_q']} | "
        f"{r['aAcc']} | {r['fAcc']} | {r['qAcc']} | {r['errors']} | `{r['model_rev']}` |"
    )

## Cell 7 (PHẦN C THẬT) — Trích xuất bảng ca lỗi 6 nhãn & tìm các ca 3 mô hình trả lời khác nhau

In [ ]:
import json
import os
import subprocess
from pathlib import Path

completed_pairs = []
for model_key, run_dir_str in RUN_DIRS.items():
    run_dir = Path(run_dir_str)
    status_data = json.loads((run_dir / "run_status.json").read_text(encoding="utf-8"))
    if status_data.get("status") == "BLOCKED":
        continue
    completed_pairs.append(f"{model_key}={run_dir}")
    print(f"\n=== TRÍCH XUẤT CA LỖI: {model_key} ===")
    subprocess.run(
        [
            "python", "-m", "benchmarks.hallusionbench.analysis",
            "--mode", "extract",
            "--run-dir", str(run_dir),
            "--dataset-json", f"{os.environ['DATA_ROOT']}/HallusionBench.json",
            "--data-root", os.environ["DATA_ROOT"],
        ],
        check=True,
    )
    md_file = run_dir / "analysis" / "error_cases.md"
    if md_file.is_file():
        print("\n--- 10 dòng đầu bảng lỗi Markdown ---")
        print("\n".join(md_file.read_text(encoding="utf-8").splitlines()[:10]))

if len(completed_pairs) >= 2:
    disagree_path = Path(os.environ["OUTPUT_DIR"]) / os.environ["EXPERIMENT_ID"] / "cross_model_disagreements.json"
    print("\n=== SO SÁNH BẤT ĐỒNG GIỮA CÁC MÔ HÌNH ===")
    subprocess.run(
        [
            "python", "-m", "benchmarks.hallusionbench.analysis",
            "--mode", "disagreements",
            "--run-dirs", *completed_pairs,
            "--dataset-json", f"{os.environ['DATA_ROOT']}/HallusionBench.json",
            "--data-root", os.environ["DATA_ROOT"],
            "--output", str(disagree_path),
        ],
        check=True,
    )
    items = json.loads(disagree_path.read_text(encoding="utf-8"))
    print(f"Tìm thấy {len(items)} ca bất đồng giữa các mô hình -> lưu tại {disagree_path}")

## Cell 8 — Gom toàn bộ kết quả (Audit + Pilot + 3 Model + Bảng lỗi) thành 1 file `.zip` & tải về máy tính

In [ ]:
import os
import zipfile
from pathlib import Path
from google.colab import files

exp_id = os.environ["EXPERIMENT_ID"]
exp_dir = Path(os.environ["OUTPUT_DIR"]) / exp_id
audit_dir = Path(os.environ["AUDIT_DIR"])
bundle_zip = Path(os.environ["OUTPUT_DIR"]) / f"hallusionbench_results_{exp_id}.zip"

with zipfile.ZipFile(bundle_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for src_dir, prefix in ((audit_dir, "audit_and_pilot"), (exp_dir, exp_id)):
        if not src_dir.exists():
            continue
        for path in sorted(src_dir.rglob("*")):
            if path.is_file():
                zf.write(path, arcname=f"{prefix}/{path.relative_to(src_dir)}")

size_kb = round(bundle_zip.stat().st_size / 1024, 1)
print(f"Đã đóng gói toàn bộ kết quả ({size_kb} KB) lưu trên Google Drive:")
print(" ->", bundle_zip)

# Bấm chạy cell này sẽ tự bật hộp thoại tải file .zip về máy tính của bạn
files.download(str(bundle_zip))